# 🎬 Studio Video GPU — free AI video on Colab

Turns this Colab into a **real AI video server** for your Studio app.

## Do exactly this
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run **Cell A** (installs). When it finishes it will **restart the session automatically** — that's normal and expected.
3. After the restart, run **Cell B**. It loads the model, starts the server, and prints your URL.
4. Copy the `https://….trycloudflare.com` URL → Studio app → **API key → GPU server URL → Save keys**

⚠️ Keep this tab open while you generate. The URL changes every time you re-run.

In [ ]:
#@title Cell A — install (auto-restarts when done)
import torch, subprocess, os, sys
assert torch.cuda.is_available(), '❌ No GPU. Runtime → Change runtime type → T4 GPU, then run this again.'
print('GPU:', torch.cuda.get_device_name(0))

# Unpinned installs: Colab's Python 3.12 already has newer torch/hub, and an old
# pinned diffusers is what caused the ImportError.
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub imageio imageio-ffmpeg fastapi 'uvicorn[standard]' nest_asyncio 2>&1 | tail -3
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

print('\n✅ Installed. Restarting the session now — this is expected.')
print('   When it comes back, run Cell B.')
# A restart is REQUIRED: pip upgraded packages Python had already imported.
os.kill(os.getpid(), 9)


In [ ]:
#@title Cell B — load model, start server, print your URL
import torch, io, tempfile, threading, subprocess, re, time
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import numpy as np, imageio, nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

DEVICE, DTYPE = 'cuda', torch.float16
BASE = 'emilianJR/epiCRealism'
REPO, CKPT = 'ByteDance/AnimateDiff-Lightning', 'animatediff_lightning_4step_diffusers.safetensors'

print('⏳ Loading model (2-3 min the first time)…')
adapter = MotionAdapter().to(DEVICE, DTYPE)
adapter.load_state_dict(load_file(hf_hub_download(REPO, CKPT), device=DEVICE))
pipe = AnimateDiffPipeline.from_pretrained(BASE, motion_adapter=adapter, torch_dtype=DTYPE).to(DEVICE)
pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
pipe.enable_vae_slicing()
print('✅ Model loaded')

def make_video(prompt: str) -> bytes:
    out = pipe(prompt=prompt, guidance_scale=1.0, num_inference_steps=4)
    frames = [np.array(f) for f in out.frames[0]]
    path = tempfile.mktemp(suffix='.mp4')
    imageio.mimsave(path, frames, fps=8, codec='libx264', output_params=['-pix_fmt','yuv420p'])
    return open(path,'rb').read()

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health(): return {'ok': True}

@app.get('/generate')
def generate(prompt: str = 'a cinematic scene'):
    try: return Response(content=make_video(prompt), media_type='video/mp4')
    except Exception as e: return JSONResponse({'error': str(e)}, status_code=500)

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
time.sleep(3)
print('✅ Server running')

box = {}
def tunnel():
    p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m and 'url' not in box: box['url'] = m.group(0)
threading.Thread(target=tunnel, daemon=True).start()
for _ in range(90):
    if 'url' in box: break
    time.sleep(1)

if 'url' in box:
    print('\n'+'='*62)
    print('🎉  YOUR GPU SERVER URL — paste into Studio → API key → GPU server URL:')
    print('\n     ' + box['url'] + '\n')
    print('='*62)
    print('⚠️  Keep this tab open. Test it: ' + box['url'] + '/health')
else:
    print('❌ Tunnel did not start — just run this cell again.')
